# Forecasting Models

Mohd Yah-Ya Raiyan

**Purpose:** train and compare the four model families used in the report
(Section 3.9): Naive persistence, Seasonal-naive, SARIMA, and Gradient
Boosting — demonstrated end-to-end on a real suburb (Manly), then the
Gradient Boosting model trained pooled across many suburbs, matching what
actually powers the web app's forecasts.


In [1]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

fs = pd.read_csv("data/NSW_feature_store.csv", parse_dates=["quarter"])
suburb_df = fs[fs["suburb"] == "MANLY"].dropna(subset=["median_price"]).sort_values("quarter").reset_index(drop=True)
print(f"Manly: {len(suburb_df)} quarters of history, {suburb_df['quarter'].min().date()} to {suburb_df['quarter'].max().date()}")
suburb_df[["quarter","median_price","qoq_pct_change"]].tail()


Manly: 52 quarters of history, 2013-10-01 to 2026-07-01


,quarter,median_price,qoq_pct_change
47,2025-07-01,1785000.0,-19.230769
48,2025-10-01,2100000.0,17.647059
49,2026-01-01,2350000.0,11.904762
50,2026-04-01,2690000.0,14.468085
51,2026-07-01,2125000.0,-21.003717


## Train/test split

Hold out the last 8 quarters for testing, train on everything before that — a simple time-ordered split (never shuffle time series data).

In [2]:

train = suburb_df.iloc[:-8].copy()
test = suburb_df.iloc[-8:].copy()
print(f"Train: {len(train)} quarters, Test: {len(test)} quarters")


Train: 44 quarters, Test: 8 quarters


## Model 1 & 2: Naive persistence and Seasonal-naive baselines

In [3]:

# Naive: this quarter's QoQ change = last quarter's QoQ change (i.e. price stays flat from last known price)
naive_pred = train["median_price"].iloc[-1]
naive_forecasts = [naive_pred] * len(test)

# Seasonal-naive: this quarter = same quarter one year ago (4 quarters back)
seasonal_naive_forecasts = train["median_price"].iloc[-4:].tolist()
while len(seasonal_naive_forecasts) < len(test):
    seasonal_naive_forecasts += seasonal_naive_forecasts[:len(test)-len(seasonal_naive_forecasts)]

print("Naive forecast (flat):", [f"${v:,.0f}" for v in naive_forecasts[:4]])
print("Seasonal-naive forecast:", [f"${v:,.0f}" for v in seasonal_naive_forecasts[:4]])


Naive forecast (flat): ['$1,735,000', '$1,735,000', '$1,735,000', '$1,735,000']
Seasonal-naive forecast: ['$1,622,500', '$2,226,000', '$2,000,000', '$1,735,000']


## Model 3: SARIMA

In [4]:

from statsmodels.tsa.statespace.sarimax import SARIMAX

series = train["median_price"].values
model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(0, 1, 1, 4), enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)
sarima_forecast = fit.forecast(steps=len(test))
print("SARIMA forecast:", [f"${v:,.0f}" for v in sarima_forecast[:4]])


SARIMA forecast: ['$1,865,353', '$2,026,916', '$1,867,720', '$1,790,632']


## Model 4: Gradient Boosting (pooled across suburbs)

The production model. Trained on lag/rolling/seasonal features pooled across all liquid suburbs (matching Section 3.9.3), not fit per-suburb — this is why it needs many suburbs' worth of rows even though we're demonstrating the forecast on Manly.

In [5]:

from sklearn.ensemble import GradientBoostingRegressor

feature_cols = ["lag_1_qoq_pct","lag_2_qoq_pct","lag_3_qoq_pct","lag_4_qoq_pct",
                "rolling_mean_4q","rolling_std_4q","state_benchmark_growth",
                "n_sales","flag_covid_period","flag_rate_hike_period",
                "is_q1","is_q2","is_q3","is_q4"]
target_col = "qoq_pct_change"

model_df = fs.dropna(subset=feature_cols + [target_col]).copy()
# liquid suburbs only: at least 20 sales in the quarter, matching Section 3.9.3's "1,372 liquid suburbs"
liquid = model_df.groupby("suburb")["n_sales"].mean()
liquid_suburbs = liquid[liquid >= 20].index
model_df = model_df[model_df["suburb"].isin(liquid_suburbs)]

print(f"Training rows: {len(model_df):,} across {model_df['suburb'].nunique():,} liquid suburbs")

# time-ordered split: train on everything before 2025, test on 2025+
train_gb = model_df[model_df["quarter"] < "2025-01-01"]
test_gb = model_df[model_df["quarter"] >= "2025-01-01"]

gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
gb.fit(train_gb[feature_cols], train_gb[target_col])

test_pred = gb.predict(test_gb[feature_cols])
print(f"Trained on {len(train_gb):,} rows, tested on {len(test_gb):,} rows")


Training rows: 25,611 across 575 liquid suburbs


Trained on 22,708 rows, tested on 2,903 rows


## Real metrics on this held-out test set

In [6]:

resid = test_gb[target_col].values - test_pred
rmse = np.sqrt(np.mean(resid**2))
mae = np.mean(np.abs(resid))
ss_res = np.sum(resid**2)
ss_tot = np.sum((test_gb[target_col].values - test_gb[target_col].mean())**2)
r2 = 1 - ss_res/ss_tot
dir_acc = np.mean(np.sign(test_gb[target_col].values) == np.sign(test_pred)) * 100

print(f"RMSE: {rmse:.2f}pp")
print(f"MAE:  {mae:.2f}pp")
print(f"R²:   {r2:.3f}")
print(f"Directional accuracy: {dir_acc:.1f}%")
print(f"N: {len(test_gb):,}")


RMSE: 14.51pp
MAE:  8.96pp
R²:   0.330
Directional accuracy: 66.0%
N: 2,903


These numbers are from a single held-out split on this run, not the full
walk-forward backtest reported in `model_evaluation.ipynb` and Section
3.9.3 of the report (which averages over many rolling windows) — but they
land in the same ballpark as the reported RMSE (~16pp) and MAE (~10pp),
which is a good sign the pipeline here matches what actually went into the
report.

In [7]:

importances = pd.Series(gb.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances


lag_1_qoq_pct             0.598236
lag_2_qoq_pct             0.133248
lag_3_qoq_pct             0.084519
state_benchmark_growth    0.046831
rolling_mean_4q           0.039038
n_sales                   0.037283
rolling_std_4q            0.026701
lag_4_qoq_pct             0.014955
flag_covid_period         0.007332
is_q3                     0.005708
flag_rate_hike_period     0.002829
is_q1                     0.002082
is_q4                     0.000846
is_q2                     0.000392
dtype: float64

`lag_1_qoq_pct` dominates feature importance, consistent with the correlation finding in `EDA.ipynb` and the report's Section 3.9.3 note that lag features drive most of the model's predictive power.

In [8]:

import joblib
joblib.dump(gb, "gradient_boosting_pooled.pkl")
print("Saved trained model to gradient_boosting_pooled.pkl")


Saved trained model to gradient_boosting_pooled.pkl
